In [ ]:
import pandas as pd
from pinecone import Pinecone
from tqdm import tqdm
import numpy as np
import ast
import os

In [ ]:
tqdm.pandas()

In [ ]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [ ]:
index = pc.Index(host='https://dinov2base-hq9ybxr.svc.aped-4627-b74a.pinecone.io')

In [ ]:
df=pd.read_csv("../data/vectors.csv")
df.info()

In [ ]:
df.dropna(subset="vectors", inplace=True)

In [ ]:
df["LowPrice"] = df["LowPrice"].astype(str)
df.fillna("None", inplace=True)

In [ ]:
def normalize_vector(vector):
    vector = np.array(ast.literal_eval(vector), dtype=float)
    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector
    return vector / norm

df["vectors"] = df["vectors"].apply(normalize_vector)

In [ ]:
metadata_columns = ["Name", "ArticleType", "MetalName", "MainStoneName", "StoneShape", "Dim1ConfigDesc", "LowPrice", "URL", "ImageURL", "SFCCMasterId"]
df["Metadata"] = df[metadata_columns].to_dict(orient="records")
modified_df = df[["StockCode", "Metadata", "vectors"]]

vector = [
    (str(row["StockCode"]), np.array(row["vectors"], dtype=np.float32).tolist(), row["Metadata"])
    for row in modified_df.to_dict(orient="records")
]

In [ ]:
vector[0]

In [ ]:
batch_size = 500

for i in tqdm(range(0, len(vector), batch_size)):
    batch = vector[i:i+batch_size]
    index.upsert(namespace="dino_inv", vectors=batch)